[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C06_Interpretability_Course/06_steering/06_steering_vectors.ipynb)

# 06 · Steering Vectors：用一根向量改变模型行为

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 numpy + matplotlib，自包含，秒级运行。

**本 notebook 你将完成：**

1. 构造一个 $d=64$ 的玩具**行为模型**：行为由某根隐方向上的投影决定，配 logistic 读出头；
2. 实现 **CAA**（contrastive activation addition）：对比对均值差提取 steering vector，验证与真方向余弦相似度 > 0.99；
3. 实现 **activation addition**：扫描强度 $\alpha$，画行为翻转率的 **dose-response 曲线**；
4. 实现 **directional ablation**（投影消除）：行为读出掉到随机，同时验证无关的"能力"读出无损；
5. **随机向量对照**：证明效果来自方向本身，而不是"任何扰动都行"；
6. 4 道 ✏️ 练习巩固核心函数（assert 自动判分）。

参考：[Turner 2023] *ActAdd* (arXiv:2308.10248) · [Panickssery 2023] *CAA* (arXiv:2312.06681) · [Zou 2023] *RepE* (arXiv:2310.01405) · [Arditi 2024] *Refusal Direction* (arXiv:2406.11717)

## 1 · 玩具行为模型：行为 = 隐方向上的投影

真实实验里，steering 作用在 LLM 第 $\ell$ 层 residual stream 上。这里我们把那一层抽象成一个有 **ground truth** 的玩具：

- 激活 $h \in \mathbb{R}^{64}$ 来自两类"语境"（想成 **拒绝 / 服从**，或 负向 / 正向情感）：
  $h = s_b \cdot \sigma \, v_{\text{beh}} + s_c \cdot \sigma \, v_{\text{cap}} + \text{noise}$，其中 $s_b, s_c \in \{-1,+1\}$ 独立；
- $v_{\text{beh}}$ 是隐藏的**行为方向**（模型"决定拒绝与否"的内部变量），$v_{\text{cap}}$ 是与之**正交**的"能力方向"（代表一项无关能力，如答题正确性）；
- 两个 **logistic 读出头**扮演"模型的下游行为"：`behavior_pred` 读行为、`capability_pred` 读能力。

关键设定：读出头用的就是真方向 $v_{\text{beh}}, v_{\text{cap}}$ ——所以**沿 $v_{\text{beh}}$ 写入必然改变行为**（因果），这给了我们检验各种 steering 方法的标准答案。实验者视角下 $v_{\text{beh}}$ 是未知的，只能从数据里估。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
D, N = 64, 2000
SIG, NOISE = 1.5, 0.6          # 信号强度 / 噪声标准差

def unit(v):
    return v / np.linalg.norm(v)

# ---- ground truth 方向（实验者视角下未知）----
v_beh = unit(rng.normal(size=D))                          # 行为方向
v_cap = rng.normal(size=D)
v_cap = unit(v_cap - (v_cap @ v_beh) * v_beh)             # 能力方向，与行为方向严格正交
assert abs(v_beh @ v_cap) < 1e-12

# ---- 合成两类语境的激活 ----
y_beh = rng.integers(0, 2, N)                             # 1 = 表现行为（如拒绝），0 = 不表现
y_cap = rng.integers(0, 2, N)                             # 无关的能力标签，独立采样
H = (SIG * (2 * y_beh - 1)[:, None] * v_beh
     + SIG * (2 * y_cap - 1)[:, None] * v_cap
     + NOISE * rng.normal(size=(N, D)))

# ---- 两个 logistic 读出头（扮演模型的下游行为）----
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def behavior_pred(acts):
    return (sigmoid(4.0 * acts @ v_beh) > 0.5).astype(int)

def capability_pred(acts):
    return (sigmoid(4.0 * acts @ v_cap) > 0.5).astype(int)

def acc(pred, y):
    return float((pred == y).mean())

print(f"行为读出 baseline 准确率: {acc(behavior_pred(H), y_beh):.4f}")
print(f"能力读出 baseline 准确率: {acc(capability_pred(H), y_cap):.4f}")
assert acc(behavior_pred(H), y_beh) > 0.95 and acc(capability_pred(H), y_cap) > 0.95

## 2 · CAA：对比对均值差 [Panickssery 2023]

CAA 的全部数学是一行：

$$ v_{\text{CAA}} = \frac{1}{N}\sum_i h(x_i^+) - \frac{1}{N}\sum_i h(x_i^-) = \mu_+ - \mu_- $$

每对差 = 行为方向 + 这一对特有的噪声；对很多对取平均，噪声以 $1/\sqrt{N}$ 速度互相抵消，行为方向被留下。
下面验证两件事：① 单对差与真方向 $v_{\text{beh}}$ 的余弦相似度很低（噪声主导）；② 均值差随对数增加迅速逼近真方向。

In [ ]:
pos, neg = H[y_beh == 1], H[y_beh == 0]          # 对比组：表现行为 / 不表现行为
v_caa = pos.mean(axis=0) - neg.mean(axis=0)       # CAA steering vector（未归一化）
v_hat = unit(v_caa)

cos_full = float(v_hat @ v_beh)
cos_single = float(unit(pos[0] - neg[0]) @ v_beh)
print(f"单对差   与真方向的余弦: {cos_single:.3f}   <- 噪声主导")
print(f"全部均值差与真方向的余弦: {cos_full:.4f}   <- 方向被恢复")
assert cos_full > 0.99

# 余弦相似度随对比对数量的收敛曲线
n_pairs_list = [1, 2, 5, 10, 20, 50, 100, 200, 500, len(neg)]
cosines = [float(unit(pos[:n].mean(axis=0) - neg[:n].mean(axis=0)) @ v_beh)
           for n in n_pairs_list]

plt.figure(figsize=(7, 4))
plt.semilogx(n_pairs_list, cosines, marker="o")
plt.axhline(1.0, color="gray", ls="--", lw=1)
plt.xlabel("number of contrast pairs N")
plt.ylabel("cosine(v_CAA, v_beh)")
plt.title("CAA: mean difference converges to the true direction")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("噪声以 1/sqrt(N) 收缩：几十对就能把方向估到 cos > 0.9。")

## 3 · Activation Addition 与 dose-response 曲线 [Turner 2023]

写入操作：$h' = h + \alpha\,\hat v$。我们对**所有"不表现行为"的样本**（$y_{\text{beh}}=0$，想成"本来会正常回答"）
沿 $\hat v_{\text{CAA}}$ 加入强度 $\alpha$，统计**行为翻转率**——多大比例的样本被读出头判成"表现行为"（如开始拒绝）。

评测规范（讲解第 7 节）要求扫描 $\alpha$ 画 **dose-response 曲线**，并同步监测**能力读出**是否受损。
一根真的因果方向应给出：翻转率单调上升的 S 形曲线 + 能力曲线纹丝不动。

In [ ]:
alphas = np.linspace(0.0, 5.0, 21)
yc_neg = y_cap[y_beh == 0]                       # 这批样本的能力标签
r_vec = unit(np.random.default_rng(42).normal(size=D))   # 随机方向对照（同为单位向量）

flip_caa, flip_rand, cap_acc = [], [], []
for a in alphas:
    flip_caa.append(behavior_pred(neg + a * v_hat).mean())     # CAA 方向翻转率
    flip_rand.append(behavior_pred(neg + a * r_vec).mean())    # 随机方向翻转率
    cap_acc.append(acc(capability_pred(neg + a * v_hat), yc_neg))  # 副作用监测

plt.figure(figsize=(7.5, 4.2))
plt.plot(alphas, flip_caa, marker="o", label="flip rate, steering with v_CAA")
plt.plot(alphas, flip_rand, marker="s", label="flip rate, random direction")
plt.plot(alphas, cap_acc, marker="^", ls="--", label="capability accuracy (side effect)")
plt.xlabel("steering strength alpha")
plt.ylabel("rate / accuracy")
plt.title("Dose-response: behavior flips, capability intact, random does nothing")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"alpha=0: 翻转率 {flip_caa[0]:.3f}  ->  alpha=5: 翻转率 {flip_caa[-1]:.3f}")
print(f"alpha=5 时能力读出准确率: {cap_acc[-1]:.4f}（基本无损）")
assert flip_caa[-1] > 0.95 and flip_rand[-1] < 0.2 and cap_acc[-1] > 0.95

## 4 · Directional Ablation：投影消除 [Arditi 2024]

消除一根方向 = 投影到它的正交补：

$$ h' = h - (\hat v^\top h)\,\hat v = (I - \hat v \hat v^\top)\,h $$

预期三件事（讲解第 6 节的几何保证）：
1. 消除后激活在 $\hat v$ 上的投影**严格为 0**（坐标被钉死，不是平移）；
2. **行为读出掉到随机**（~50%）——行为自由度被删除，对应 refusal direction 消融后"模型不再会拒绝"；
3. **能力读出无损**——$v_{\text{cap}} \perp v_{\text{beh}}$，正交方向分毫不动。这就是"删行为不删能力"的几何本质，也是它对安全评测构成冲击的原因。

In [ ]:
def ablate(acts, v):
    u = unit(v)
    return acts - (acts @ u)[:, None] * u

H_ab = ablate(H, v_caa)

print(f"消除后 |投影| 最大值      : {np.abs(H_ab @ v_hat).max():.2e}   (严格为 0)")
print(f"行为读出: {acc(behavior_pred(H), y_beh):.4f} -> {acc(behavior_pred(H_ab), y_beh):.4f}   (掉到随机)")
print(f"能力读出: {acc(capability_pred(H), y_cap):.4f} -> {acc(capability_pred(H_ab), y_cap):.4f}   (无损)")

# 幂等性：投影是 projection，消一次就干净，再消无变化
assert np.allclose(ablate(H_ab, v_caa), H_ab)
assert np.abs(H_ab @ v_hat).max() < 1e-9
assert abs(acc(behavior_pred(H_ab), y_beh) - 0.5) < 0.1
assert acc(capability_pred(H_ab), y_cap) > 0.95
print("幂等性 ✓  行为删除 ✓  能力保持 ✓")

## 5 · 反例对照：随机向量什么都做不了

评测三件套的最后一件（讲解第 7 节）：**对照基线**。如果随机单位向量也能翻转行为/删除行为，
那前面的结果只说明"模型怕扰动"，而不是"我们找到了行为方向"。
随机方向与 $v_{\text{beh}}$ 的重叠期望 $\sim 1/\sqrt{d} \approx 0.125$，远不足以撬动 $\pm 1.5$ 的信号。

In [ ]:
rng_ctrl = np.random.default_rng(7)
rows = []
for k in range(5):                                # 5 根独立随机方向
    r = unit(rng_ctrl.normal(size=D))
    rows.append((k,
                 float(abs(r @ v_beh)),                                  # 与真方向的重叠
                 float(behavior_pred(neg + 5.0 * r).mean()),             # alpha=5 翻转率
                 acc(behavior_pred(ablate(H, r)), y_beh)))               # 消融后行为读出

print(f"{'随机方向':<8}{'|cos(r, v_beh)|':>16}{'steer翻转率(a=5)':>18}{'消融后行为acc':>15}")
for k, c, f, a_ in rows:
    print(f"#{k:<7}{c:>16.3f}{f:>18.3f}{a_:>15.3f}")

print(f"\n对照: v_CAA 方向 steer(a=5) 翻转率 {behavior_pred(neg + 5.0 * v_hat).mean():.3f}，"
      f"消融后行为 acc {acc(behavior_pred(ablate(H, v_caa)), y_beh):.3f}")
assert all(f < 0.3 for _, _, f, _ in rows)        # 随机 steering 几乎无效
assert all(a_ > 0.9 for _, _, _, a_ in rows)      # 随机消融不伤行为读出
print("结论：效果来自方向本身。一根向量有效，是因为它指对了地方。")

---
## ✏️ 练习 1：实现 `caa_vector(pos_acts, neg_acts)`

实现 CAA 提取器：输入两组激活矩阵（形状 `(n, d)`），返回**单位化**的均值差向量
$\hat v = \mathrm{unit}(\mu_+ - \mu_-)$。

**提示**：`pos_acts.mean(axis=0)` 两行解决，归一化用上文的 `unit`。注意方向性：交换两组输入应得到**反向**的向量（这正是 CAA "加正向、减反向"对称性的来源）。3 行以内。

In [ ]:
def caa_vector(pos_acts, neg_acts):
    # TODO: 返回单位化的均值差向量 unit(mean(pos) - mean(neg))
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
v1 = caa_vector(H[y_beh == 1], H[y_beh == 0])
assert v1.shape == (D,)                                   # 形状正确
assert abs(np.linalg.norm(v1) - 1.0) < 1e-9               # 已单位化
assert float(v1 @ v_beh) > 0.95                           # 与真方向高度对齐
assert np.allclose(caa_vector(H[y_beh == 0], H[y_beh == 1]), -v1)   # 交换输入 -> 反向
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `add_steering` 与 `flip_rate`，并验证单调性

两个函数：
1. `add_steering(acts, v, alpha)`：把 `v` **先归一化为单位向量**，再对每行激活加 $\alpha\,\hat v$，返回新矩阵（**不修改原数组**）；
2. `flip_rate(acts, v, alpha)`：对 `acts`（默认传入 $y_{\text bh}=0$ 的那批）施加 steering 后，用 `behavior_pred` 统计被判为 1 的比例。

**提示**：广播 `acts + alpha * unit(v)` 一行即可；`flip_rate` 就是 `behavior_pred(...).mean()`。逐样本看，翻转条件是 $\alpha\,(\hat v^\top v_{\text{beh}}) + \hat v_{\text{beh}}^\top h > 0$ ——对固定样本它随 $\alpha$ 单调，所以整体翻转率必须随 $\alpha$ 非降（自测会查）。合计 5 行以内。

In [ ]:
def add_steering(acts, v, alpha):
    # TODO: 返回 acts + alpha * unit(v)，不修改原数组
    raise NotImplementedError

def flip_rate(acts, v, alpha):
    # TODO: steering 后 behavior_pred 判为 1 的比例（float）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
out = add_steering(neg, 2.0 * v_beh, 1.0)                 # v 要先归一化：2v 和 v 等效
assert out.shape == neg.shape
assert np.allclose(out - neg, np.broadcast_to(v_beh, neg.shape), atol=1e-9)
assert np.allclose(add_steering(neg, v_beh, 0.0), neg)    # alpha=0 是恒等
rates = [flip_rate(neg, v_caa, a) for a in [0.0, 1.0, 2.0, 3.0, 5.0]]
assert all(rates[i] <= rates[i + 1] + 1e-12 for i in range(4))   # dose-response 单调
assert rates[0] < 0.05 and rates[-1] > 0.95               # 从基本不翻 -> 几乎全翻
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 `directional_ablation(acts, v)`

实现投影消除：把 `v` 归一化为 $\hat v$，返回 $(I - \hat v\hat v^\top)\,\text{acts}$（逐行投影到正交补，不修改原数组）。

**提示**：不要真的构造 $64\times 64$ 矩阵——`acts - (acts @ u)[:, None] * u` 一行完成。
三个必须满足的几何性质（自测全会查）：消除后在 $\hat v$ 上投影为 0；**幂等**（再消一次无变化）；
与 $\hat v$ **严格正交**的方向上坐标**逐元素不变**（自测用真方向 $v_{\text{beh}}$ 消融来检验 $v_{\text{cap}}$ 不变——
注意估计出的 $\hat v_{\text{CAA}}$ 与 $v_{\text{cap}}$ 并非严格正交，会留下 $\sim 10^{-2}$ 量级的微小泄漏，这本身就是个教训）。3 行以内。

In [ ]:
def directional_ablation(acts, v):
    # TODO: 返回 (I - u u^T) acts，u = unit(v)；不构造 64x64 矩阵
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
A = directional_ablation(H, v_caa)
u = unit(v_caa)
assert A.shape == H.shape
assert np.abs(A @ u).max() < 1e-9                          # 消除后投影严格为 0
assert np.allclose(directional_ablation(A, v_caa), A)      # 幂等：P^2 = P
A_true = directional_ablation(H, v_beh)                    # 用真方向消融：v_cap 与之严格正交
assert np.allclose(A_true @ v_cap, H @ v_cap, atol=1e-9)   # 严格正交方向坐标不变
assert abs(acc(behavior_pred(A), y_beh) - 0.5) < 0.1       # 行为读出掉到随机
print("✅ 练习 3 通过")

## ✏️ 练习 4：实现副作用度量 `side_effect_report`

把讲解第 7 节的"行为改变 vs 能力损伤成对报告"写成函数。
`side_effect_report(acts, yb, yc, v)`：对 `acts` 做 `v` 方向的 directional ablation，返回 dict：

```python
{"behavior_acc_before": ..., "behavior_acc_after": ...,
 "capability_acc_before": ..., "capability_acc_after": ...,
 "selectivity": 行为下降量 / max(能力下降量, 1e-3)}
```

**提示**：复用练习 3 的 `directional_ablation` 和上文的 `behavior_pred` / `capability_pred` / `acc`。
`selectivity` 越大说明干预越"外科手术"：对真方向应该很大（行为掉一半、能力不掉），对随机方向接近 0（行为根本不掉）。约 8 行。

In [ ]:
def side_effect_report(acts, yb, yc, v):
    # TODO: ablation 前后分别测 behavior/capability 读出准确率，组装 dict
    #       selectivity = (behavior_before - behavior_after) / max(capability_before - capability_after, 1e-3)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
rep = side_effect_report(H, y_beh, y_cap, v_caa)
keys = {"behavior_acc_before", "behavior_acc_after",
        "capability_acc_before", "capability_acc_after", "selectivity"}
assert keys <= set(rep)
assert rep["behavior_acc_before"] > 0.95
assert abs(rep["behavior_acc_after"] - 0.5) < 0.1          # 行为被删除
assert abs(rep["capability_acc_after"] - rep["capability_acc_before"]) < 0.02   # 能力无损
assert rep["selectivity"] > 100                            # 外科手术级：行为掉 ~0.5，能力几乎不掉
rep_r = side_effect_report(H, y_beh, y_cap,
                           np.random.default_rng(123).normal(size=D))
assert rep_r["behavior_acc_after"] > 0.9                   # 随机方向：行为根本不掉
assert rep_r["selectivity"] < rep["selectivity"]
print("✅ 练习 4 通过")
print(f"   v_CAA  selectivity = {rep['selectivity']:.1f}")
print(f"   随机方向 selectivity = {rep_r['selectivity']:.1f}")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def caa_vector(pos_acts, neg_acts):
    return unit(pos_acts.mean(axis=0) - neg_acts.mean(axis=0))

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def add_steering(acts, v, alpha):
    return acts + alpha * unit(v)

def flip_rate(acts, v, alpha):
    return float(behavior_pred(add_steering(acts, v, alpha)).mean())

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def directional_ablation(acts, v):
    u = unit(v)
    return acts - (acts @ u)[:, None] * u

In [ ]:
# 练习 4 参考答案（先自己做，再对照）
def side_effect_report(acts, yb, yc, v):
    after = directional_ablation(acts, v)
    bb, ba = acc(behavior_pred(acts), yb), acc(behavior_pred(after), yb)
    cb, ca = acc(capability_pred(acts), yc), acc(capability_pred(after), yc)
    return {"behavior_acc_before": bb, "behavior_acc_after": ba,
            "capability_acc_before": cb, "capability_acc_after": ca,
            "selectivity": (bb - ba) / max(cb - ca, 1e-3)}

---
## 🎯 真实数据胶囊题：真实 embedding 上的 steering 向量（数字方向）

steering 向量 = 两类激活的均值差；把它加到一个表示上能把表示朝某概念方向推。用真实 GPT-2 embedding 构造“数字 − 字母”方向，加到字母 token 上，验证数字 probe 分上升。

> 本模块新增的**真实数据**练习：用**真实 GPT-2 权重/embedding**把本章的可解释性技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, struct, urllib.request
import numpy as np
CACHE=os.path.expanduser("~/.interp_data"); os.makedirs(CACHE,exist_ok=True)
ST="https://huggingface.co/openai-community/gpt2/resolve/main/model.safetensors"
def _rng(s,e):
    req=urllib.request.Request(ST, headers={"Range":f"bytes={s}-{e}"})
    return urllib.request.urlopen(req,timeout=60).read()
def gpt2_emb_block(n=6000):
    cache=os.path.join(CACHE,f"wte_{n}.npy")
    if os.path.exists(cache): return np.load(cache)
    hlen=struct.unpack("<Q", _rng(0,7))[0]; hdr=json.loads(_rng(8,8+hlen-1))
    info=hdr["wte.weight"]; base=8+hlen; s0=info["data_offsets"][0]; d=info["shape"][1]
    raw=_rng(base+s0, base+s0+n*d*4-1)
    E=np.frombuffer(raw,dtype=np.float32).reshape(n,d).copy()
    np.save(cache,E); return E
def gpt2_vocab():
    p=os.path.join(CACHE,"vocab.json")
    if not os.path.exists(p): urllib.request.urlretrieve("https://huggingface.co/openai-community/gpt2/resolve/main/vocab.json",p)
    return json.load(open(p))
def digit_letter_dataset(lim=6000):
    "返回 (X[token嵌入], y[1=数字 0=字母], E, ids_digit, ids_alpha)"
    v=gpt2_vocab(); E=gpt2_emb_block(lim)
    dig=[i for t,i in v.items() if i<lim and t.isdigit()]
    alpha=[i for t,i in v.items() if i<lim and t.isalpha() and t.isascii()]
    rng=np.random.default_rng(0); alpha=list(rng.permutation(alpha)[:len(dig)])
    ids=dig+alpha; y=np.array([1]*len(dig)+[0]*len(alpha))
    return E[ids], y, E, dig, alpha
def shakespeare():
    p=os.path.join(CACHE,"shake.txt")
    if not os.path.exists(p): urllib.request.urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",p)
    return open(p).read()

X,y,E,dig,alpha=digit_letter_dataset()
mu,sd=X.mean(0),X.std(0)+1e-8
w=np.zeros(X.shape[1]); b=0.0; Xs=(X-mu)/sd
for _ in range(600):
    p=1/(1+np.exp(-(Xs@w+b))); g=p-y; w-=0.5*Xs.T@g/len(y); b-=0.5*g.mean()
def digit_score(vec): return float(1/(1+np.exp(-(((vec-mu)/sd)@w+b))))

**练习**：实现 `steering_vector(E, pos_ids, neg_ids)` = `mean(E[pos]) - mean(E[neg])`，和 `apply_steer(vec, sv, alpha)` = `vec + alpha*sv`。验证给字母 token 加数字方向后数字分上升。

In [ ]:
def steering_vector(E, pos_ids, neg_ids):
    # TODO: mean(E[pos]) - mean(E[neg])
    raise NotImplementedError
def apply_steer(vec, sv, alpha=4.0):
    # TODO: vec + alpha*sv
    raise NotImplementedError


In [ ]:
# 自测
sv=steering_vector(E, dig, alpha)
before=np.mean([digit_score(E[a]) for a in alpha[:30]])
after =np.mean([digit_score(apply_steer(E[a], sv, 4.0)) for a in alpha[:30]])
assert after > before, "加数字 steering 向量后字母 token 数字分应上升"
# alpha 越大推得越多
assert digit_score(apply_steer(E[alpha[0]],sv,8.0)) >= digit_score(apply_steer(E[alpha[0]],sv,2.0))
print(f"steering ✓  字母token数字分 {before:.3f} -> 加向量后 {after:.3f}")


### 📖 参考答案

In [ ]:
def steering_vector(E, pos_ids, neg_ids):
    return E[pos_ids].mean(0) - E[neg_ids].mean(0)
def apply_steer(vec, sv, alpha=4.0):
    return vec + alpha*sv
print("✓ steering 向量(CAA): 均值差方向，加到激活上即可控制行为(如 refusal 方向)")